<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Solution: Few labels with embeddings

**[Few labels with embeddings](https://learning.nextia-ai.com/courses/text-routing/m04/few-labels-with-embeddings/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 4, lesson 2 of 6 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** classify messages when each route has only a few labelled examples: with the nearest route centre, with logistic regression on embeddings, and with SetFit-style training. You will see when this beats TF-IDF.

| | |
|---|---|
| **Time** | About 30 minutes. The model download is about 470 MB, once (the same model as the last lesson). |
| **Needs** | An internet connection for the setup cells and the model download from Hugging Face. No account and no key. |
| **You should know** | Sentence embeddings and cosine similarity, from [Sentence embeddings for classification](https://learning.nextia-ai.com/courses/text-routing/m04/sentence-embeddings-for-classification/). |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, sentence-transformers 6.2.0, SetFit 1.2.0 and PyTorch 2.15.0 (CPU), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m04/few-labels-with-embeddings/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M04-L02-few-labels-with-embeddings/few-labels-with-embeddings-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, sentence-transformers 6.2.0, torch 2.15.0, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2 sentence-transformers==6.2.0 torch==2.15.0
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import sentence_transformers; print("sentence-transformers", sentence_transformers.__version__)
import torch; print("torch", torch.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield inbox of this course (about 9 MB) into a folder `larkfield-inbox/data`, and the course's helper `routing.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 10000 messages (6000 train, 2000 validation, 2000 test), all synthetic.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-inbox` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield inbox is synthetic: we made it for this course. It has no real people. The answers of the hosted models were recorded once, so you need no key.

In [ ]:
# Setup: download the course data into larkfield-inbox/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/"
FILES = {
    "data/corrections.csv": "be3088aecd9b7ba5d07ea222b79e06a476806eba81bc80c2f33da695878e5cc0",
    "data/costs.json": "9a2d566f062c3382b8884ef07756f886179c69a3525d80e3d4c15c0c4fd4507e",
    "data/dataset-card.md": "ec1d7769d1d68a9e51a5b12660364c74352af936df79533a0d2170867171bdd1",
    "data/double_labelled.csv": "ba4a8dadc3c1ba8d48af893124bd33fb7a895f917a2add211486e3dbd353dfc9",
    "data/guidelines.md": "04df0cd72da4e6b7e6aa629ca21205e5395df7b66aae9268618ae14597e45782",
    "data/launch_labels_sample.csv": "600c689325b7c14e56f4a27e92e395eb0b2e965411b1abf9d724720c8b485029",
    "data/launch_month.csv": "7ef3cc2c5fa910023c6330c1e895744358846ce451e551b86be8c9abe1c4289f",
    "data/messages.csv": "266ca23c0551249fc114fc74253103ea8602a04be1fe31d1b7f91bc94ce6dd5e",
    "data/old_queue.csv": "e3cde5d19d3ebe2e6e0e067693c76796c0abbf89df0dc93dc310ad3fe42a5cae",
    "data/predictions/embeddings-newer.csv": "3c7ffe5b5371ca60c3b6bedc17b307142bcfd77f06c97db7ed267b703a50a8d2",
    "data/predictions/embeddings-newer.json": "dcd537f2971b9a48b5354cc9fbb48b87689b83b3a479da9acd119701c2fc19f6",
    "data/predictions/embeddings-older.csv": "a06c490ac17cbc1c3c75380f6cdf7775483de92961cdbb78b2a7914256926539",
    "data/predictions/embeddings-older.json": "db658bbd13861529122807726b6b41e1da9c70e3b55aba55cce9e868a795e8b4",
    "data/predictions/encoder-finetuned.csv": "abced2276671bfd413319573907a48c60952310e50978dba96c0dde0332b7d39",
    "data/predictions/encoder-finetuned.json": "1f7c89271a199c1d659bdc29d8ac0592558a1424218bd0519bbbde9840212b94",
    "data/predictions/gliclass-zero-shot.csv": "2c22e5036de94a376f7c0ccf22603c4d7c624135c8a4965123c9492748814008",
    "data/predictions/gliclass-zero-shot.json": "95ad2acafd48ad2c24516333dba7e78a3eabc7ca616c4053ce2f144f56cbaaec",
    "data/predictions/llm-few-shot.csv": "bcdaee3550f78ebb672465314bbcc9ad26283aaea69265a979739cf6a5f965e2",
    "data/predictions/llm-few-shot.json": "104b446894a0450a7e02fcd07c6db44f3e1a5486002b388374ce6dae4a744a12",
    "data/predictions/llm-zero-shot.csv": "5518e2b668a110b572687313bc13bd689774dff96c0a1614e1cd92e14ddf9ce7",
    "data/predictions/llm-zero-shot.json": "9a72ce600d0d858b00dd3b37247a305f9e8b3fc2fef3b8dbe60c0871b0c1b355",
    "data/predictions/local-llm-prompted.csv": "5bb0d84f8ee9b5086818d3525e1da3f8172b38b7875b10d1d3021b3ad5889f33",
    "data/predictions/local-llm-prompted.json": "ae328379d5ea294b3b59845ee143f24776787e748bc14c165b5e8e9ed4ff7158",
    "data/predictions/local-llm-scores.csv": "f85f570075ea9288fd14a6c3955d4310ba740814f3de472e8a263cd4f9682a5f",
    "data/predictions/local-llm-scores.json": "67a7e07ce065934a947f59c1c36d416eec76f8827f01500ce82ba6e6c4d9b99e",
    "data/predictions/nli-zero-shot.csv": "ecafb001370e5cf2904ce685683beafc86a2c290cd1da80438abb9912d099d3c",
    "data/predictions/nli-zero-shot.json": "54c2e724cb3c855a0fa6302d4ac0ab2df2ffaf788eec4ab198c6feb8d3508928",
    "data/prices.json": "9e98f9056d9d1e53d76a039da9b10d2b4fa54c8a5b35eb6cae32a2abdee7af38",
    "data/recordings/README.md": "d99eb50666045cc2ccf596e100103244fe1513b488fce46e26462a2666053ffc",
    "data/recordings/llm-few-shot.jsonl": "259a93f1d6dcbcb167cd45796c8e09a914e56c7741f14bd0aee53c996e8349d4",
    "data/recordings/llm-labels.jsonl": "cd6c670a23ee78cd3d8b51e5907ebae9d86448aa6236767c11d55227616fa8cc",
    "data/recordings/llm-reasoning.jsonl": "fc8ae631121f3240ce378df429af767ac52ea6e1560aa409de09ffc99b092300",
    "data/recordings/llm-zero-shot.jsonl": "957edfb8811e177c83bf23772f0da561659cf50728ffc95597f85cd7680aed86",
    "data/rules.json": "6f95d5e7b87869ec7975b198a3777cdd96a0db6a869554d922889479c87b2066",
    "routing.py": "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-inbox").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-inbox and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
import csv
with open("data/messages.csv", encoding="utf-8") as f:
    splits = [row["split"] for row in csv.DictReader(f)]
print(f"Ready: {len(splits)} messages ({splits.count('train')} train, {splits.count('validation')} validation, "
      f"{splits.count('test')} test), all synthetic.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course helper as `rt`, reads the three parts of the inbox (`train`, `validation`, `test`), and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

messages = pd.read_csv("data/messages.csv")
train = messages[messages["split"] == "train"].reset_index(drop=True)
validation = messages[messages["split"] == "validation"].reset_index(drop=True)
test = messages[messages["split"] == "test"].reset_index(drop=True)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: SetFit

Section 5 uses the SetFit package. Run the next cell to install it. You should see `setfit 1.2.0`.

In [ ]:
%pip install -q --disable-pip-version-check setfit==1.2.0 accelerate==1.15.0
import setfit; print("setfit", setfit.__version__)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A tiny example: the nearest centre

Each route has two labelled messages. Each message has an embedding of only two numbers here, so that you can check every step by hand. A real model gives 384 numbers. These numbers are made up for the example.

The **centre** of a route is the average of its embeddings. A new message goes to the route whose centre is the most similar (the largest cosine similarity).

> **Predict.** The new message has the numbers (0.8, 0.3). Which route wins?

In [ ]:
import numpy as np
examples = {"delivery": [[0.9, 0.1], [0.7, 0.3]], "billing": [[0.2, 0.9], [0.0, 0.8]]}
new = np.array([0.8, 0.3])

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

for route, vectors in examples.items():
    centre = np.mean(vectors, axis=0)
    print(route, "centre", centre.round(2), "cosine", round(cosine(new, centre), 3))

> **Check.** The delivery centre is (0.8, 0.2) and the billing centre is (0.1, 0.85). The new message goes to **delivery**. Change `new` to `[0.1, 0.7]`: it goes to **billing**.

## 2. Embed the inbox

The same frozen model as in the last lesson. It takes about a minute on a laptop CPU.

In [ ]:
import time, torch
from sentence_transformers import SentenceTransformer
torch.set_num_threads(4)
torch.manual_seed(0)
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", revision="e8f8c211226b894fcb81acc59f3b34ba3efd5f42", device="cpu")
t0 = time.time()
E_train = embedder.encode(train["text"].tolist(), batch_size=64, normalize_embeddings=True)
E_test = embedder.encode(test["text"].tolist(), batch_size=64, normalize_embeddings=True)
print(E_train.shape, E_test.shape, f"{time.time() - t0:.0f} seconds")

> **Check.** You should see `(6000, 384) (2000, 384)`.

## 3. Eight labels per route

Imagine a new team: Chloé labels only **8 messages per route**, 56 in all. The function below draws them at random, with a fixed seed so that you get the same messages as the course. Then three methods learn from the same 56 messages:

- **TF-IDF + logistic regression** (Module 3), on the words;
- **embeddings + logistic regression**, on the 384 numbers;
- **nearest centre**, as in section 1, on the 384 numbers.

> **Predict.** Which of the three is best with 56 labels? Which is best with all 6,000 (you know this one from the last lesson)?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, make_pipeline

ROUTES = rt.ROUTES
truth = test["route"].to_numpy()

def draw(k, repeat=0):
    """k training messages per route, at random (the same draw as the course)."""
    rng = np.random.default_rng([36, k, repeat])
    rows = []
    for r in ROUTES:
        rows.extend(rng.choice(np.flatnonzero(train["route"].to_numpy() == r), size=k, replace=False))
    return np.array(sorted(rows))

def tfidf_model():
    return make_pipeline(
        FeatureUnion([("words", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")),
                      ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True))]),
        LogisticRegression(C=10, max_iter=3000))

def nearest_centre(idx):
    labels = train["route"].to_numpy()[idx]
    centres = np.stack([E_train[idx][labels == r].mean(axis=0) for r in ROUTES])
    return np.array(ROUTES)[np.argmax(E_test @ centres.T, axis=1)]

def three_methods(idx):
    sub = train.iloc[idx]
    acc_tfidf = (tfidf_model().fit(sub["text"], sub["route"]).predict(test["text"]) == truth).mean()
    acc_emb = (LogisticRegression(C=10, max_iter=3000).fit(E_train[idx], sub["route"]).predict(E_test) == truth).mean()
    acc_near = (nearest_centre(idx) == truth).mean()
    return round(float(acc_tfidf), 3), round(float(acc_emb), 3), round(float(acc_near), 3)

idx8 = draw(8)
print(len(idx8), "labelled messages")
print("TF-IDF + LR, embeddings + LR, nearest centre:", three_methods(idx8))

> **Check.** You should see `56 labelled messages` and `(0.51, 0.632, 0.525)`.

With 56 labels, the embeddings win clearly: TF-IDF has seen too few words, but the embedding model already knows that "parcel", "colis" and "delivery" are related.

## 4. Your turn: 32 labels per route

> **Your turn.** Draw 32 messages per route with `draw(32)`, and compute the accuracy of the **nearest centre** method on the test messages. Put it in `near32`, rounded to 3 decimals.

In [ ]:
idx32 = draw(32)
near32 = round(float((nearest_centre(idx32) == truth).mean()), 3)
print(near32, three_methods(idx32))

In [ ]:
expect('near32', near32, 0.665, tolerance=0.0005)

The nearest centre gains little from more labels: one average per route is a simple summary. Logistic regression keeps learning, and TF-IDF catches up. The lesson page has the curve for 4 to 64 labels per route, with 5 draws each.

## 5. SetFit-style training with 8 labels per route

**SetFit** fine-tunes the embedding model itself with pairs of the few labelled messages: two messages of the same route should get closer, two of different routes further apart. Then it trains logistic regression on top. It takes about half a minute here.

In [ ]:
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
sub = train.iloc[idx8]
sf = SetFitModel.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", revision="e8f8c211226b894fcb81acc59f3b34ba3efd5f42", device="cpu")
args = TrainingArguments(batch_size=16, num_epochs=1, num_iterations=10, seed=36)
trainer = Trainer(model=sf, args=args, train_dataset=Dataset.from_dict({"text": list(sub["text"]), "label": list(sub["route"])}))
t0 = time.time()
trainer.train()
print(f"trained in {time.time() - t0:.0f} seconds")
setfit_acc = round(float((np.array(sf.predict(list(test["text"]))) == truth).mean()), 3)
print("SetFit, 8 per route:", setfit_acc)

> **Check.** You should see about 0.63. On this draw, SetFit is no better than logistic regression on the frozen embeddings (0.632). It costs training time and gives nothing here. Measure before you add a step.

## 6. Failure case: one draw is luck

With 4 labels per route, which 4 messages you draw matters a lot. The next cell tries 5 different draws.

In [ ]:
for repeat in range(5):
    print("draw", repeat, three_methods(draw(4, repeat)))

> **Check.** The embeddings + logistic regression accuracy goes from 0.514 to 0.606 between draws: almost 10 points from luck alone.

With few labels, one result is not a result. Try several draws, and report the range, not the best one.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m04/few-labels-with-embeddings/) has the curve of all three methods from 4 to 64 labels per route and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Fine-tune a small encoder on CPU](https://learning.nextia-ai.com/courses/text-routing/m04/fine-tune-a-small-encoder-on-cpu/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m04/few-labels-with-embeddings/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.